<a href="https://colab.research.google.com/github/vzehraz/Eureka-Service/blob/main/server_app_final_gpt_5_with_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# MODULE IMPORTS & STANDARD LIBRARY SETUP
# ==========================================

# 'os' provides a way of using operating system dependent functionality like environment variable fetching.
import os
# 'json' allows parsing of JSON API payloads and serialization of Python objects.
import json
# 'logging' is used to record runtime events, warnings, errors, and diagnostic traces.
import logging
# 'openai' is the legacy library interface for communicating with OpenAI language services.
import openai
# 'time' is utilized for measuring elapsed runtime intervals and setting token expiration values.
import time
# 'jwt' (PyJWT) handles sign, encode, and decode verification operations on JSON Web Tokens.
import jwt
# 'requests' enables sending HTTP/1.1 requests to external service endpoints (e.g. Firebase Auth API).
import requests
# 'firebase_admin' enables programmatic server-side SDK control over Firebase Cloud Resources.
import firebase_admin
# 'uuid' generates RFC 4122 compliant universally unique identifiers for chats and questions.
import uuid
# 'types' and 'pickle' assist with runtime object configuration handling and serialization operations.
import types, pickle

# Flask core framework components for handling routing, session states, file transfers, and request abortion.
from flask import Flask, render_template, request, jsonify, session, send_file, abort
# Response structures supporting streaming contexts to client listeners in real-time.
from flask import Response, stream_with_context
# dotenv parses keys from a local '.env' file into Python's execution environment dictionary.
from dotenv import load_dotenv
# PdfReader extracts readable text streams from static document files.
from pypdf import PdfReader
# CharacterTextSplitter splits unstructured text records into small digestible semantic chunks.
from langchain_text_splitters import CharacterTextSplitter
# OpenAIEmbeddings translates natural text streams into multi-dimensional numerical vector representations.
from langchain_openai import OpenAIEmbeddings
# FAISS manages offline vector database index creation, storage, and similarity search lookup operations.
from langchain_community.vectorstores import FAISS
# ConversationBufferMemory tracks local chat transcript structures inside LangChain executions.
from langchain_classic.memory import ConversationBufferMemory
# ConversationalRetrievalChain binds LLMs with retrievers and history memory tracks to manage Q&A logic.
from langchain_classic.chains import ConversationalRetrievalChain
# HumanMessage and AIMessage represent client query inputs and machine response structures respectively.
from langchain_core.messages import HumanMessage, AIMessage
# ChatOpenAI acts as the connector class for interacting with OpenAI Gpt endpoints.
from langchain_openai import ChatOpenAI
# Firebase authorization, credentials, and firestore database wrappers.
from firebase_admin import credentials, firestore, auth
# Flask extension wrapper resolving strict Cross-Origin Resource Sharing network constraints.
from flask_cors import CORS
# Google's OAuth 2.0 validation component used to safely analyze sign-in identity claims.
from google.oauth2 import id_token
# Secure request transport agent for executing identity lookup validations.
from google.auth.transport import requests as google_requests
# ChatMessageHistory maintains structural logs of conversation context tracks in active environments.
from langchain_community.chat_message_histories import ChatMessageHistory
# BaseMessage is the parent typing standard from which all message entities inherit.
from langchain_core.messages import BaseMessage
# Helper utility for requesting template generation layouts from Anthropic Claude LLM models.
from claude_template import output_from_claude
# SelectingBestNiche maps client queries to find the most contextually relevant internal database directories.
from finding_actual_rag import SelectingBestNiche

# ==========================================
# FIREBASE & DATABASE INITIALIZATION
# ==========================================

# Path loading of private Firebase credential configuration JSON files.
cred = credentials.Certificate("firebase_config.json")
# Initialization statement mapping authorization access profiles onto the Firebase runtime.
firebase_admin.initialize_app(cred)
# 'db' holds the connected client interface object for managing Google Firestore NoSQL collections.
db = firestore.client()

# ==========================================
# DIAGNOSTICS & LOGGER CONFIGURATION
# ==========================================

# Registering an execution logger named 'app_logger' to trace runtime actions.
logger = logging.getLogger('app_logger')
# Configures diagnostic filters to track everything down to verbose DEBUG level signals.
logger.setLevel(logging.DEBUG)

# ==========================================
# ENVIRONMENT VARIABLES CONFIGURATION
# ==========================================

# Locates and parses keys defined in our '.env' environment file into system configurations.
load_dotenv(dotenv_path=".env")
# 'api_key' captures the OpenAI validation token used to authorize vector generation and generation calls.
api_key = os.getenv("OPENAI_API_KEY")
# 'firebase_web_api_key' stores client verification secrets needed to access Firebase Auth REST services.
firebase_web_api_key = os.getenv("FIREBASE_WEB_API_KEY")
# 'google_client_id' keeps track of registration client tags verified by Google Sign-In protocols.
google_client_id = os.getenv("GOOGLE_CLIENT_ID")
# 'microsoft_client_id' retains application credential handles defined inside Azure portals.
microsoft_client_id = os.getenv("MICROSOFT_CLIENT_ID")
# 'microsoft_client_secret' houses the cryptographic private key belonging to Microsoft Azure credentials.
microsoft_client_secret = os.getenv("MICROSOFT_CLIENT_SECRET")

# ==========================================
# SYSTEM CACHES & DATA DICTIONARY MAPS
# ==========================================

# 'vector_cache' holds loaded vector database index mappings to avoid costly disk re-reads.
vector_cache = dict()

# 'filename_map' translates RAG system keys into corresponding database directories stored on disk.
filename_map = {
    "r1": "vector_store_qa",
    "r2": "vector_store_iso_42001",  # Reference to ISO database index folder.
    "r3": "llm_bootcamp_data",
    "r4": "vector_store_foundation",  # Reference to AI Foundation index folder.
    "r5": "vector_store_practitioner"
}

# 'rag_source_map' maps client selection strings to internal system identifier codes.
rag_source_map = {
    "EU AI Act": "r1",
    "ISO-42001": "r2",
    "LLM Bootcamp": "r3",
    "Foundation": "r4",
    "Practitioner": "r5"
}

# ==========================================
# APP SYSTEM VALIDATIONS & DIAGNOSTICS
# ==========================================

# Verify OpenAI key exists; log error if missing, otherwise configure global OpenAI credential state.
if not api_key:
    print("ERROR: OPENAI_API_KEY is missing! Make sure .env file is configured correctly.")
else:
    openai.api_key = api_key

# Verify Firebase web API key availability for auth service validations.
if not firebase_web_api_key:
    print("ERROR: FIREBASE_WEB_API_KEY is missing! Add it to your .env file.")

# Warn if client identity integrations for Google SSO are unconfigured.
if not google_client_id:
    print("WARNING: GOOGLE_CLIENT_ID is missing! Google OAuth will not work.")

# Warn if authentication settings for Microsoft Azure SSO are unconfigured.
if not microsoft_client_id:
    print("WARNING: MICROSOFT_CLIENT_ID is missing! Microsoft OAuth will not work.")

# Warn if identity security secrets for Microsoft Azure SSO are unconfigured.
if not microsoft_client_secret:
    print("WARNING: MICROSOFT_CLIENT_SECRET is missing! Microsoft OAuth will not work.")

# ==========================================
# LARGE LANGUAGE MODEL INSTANCES SETUP
# ==========================================

# 'rag_llm' is configured as the high-precision retrieval validator model (low temperature prevents hallucination).
rag_llm = ChatOpenAI(model="gpt-4o",
                     temperature=0.2,
                     top_p=0.5
                     )

# 'gen_llm' serves as our generative platform model for formulating detailed contextual expansions.
gen_llm = ChatOpenAI(
    model="gpt-5-nano",
)

# Capture system property identifiers directly from runtime configurations.
gpt_5_nano_model = gen_llm.model_name
gpt_4_model = rag_llm.model_name

# ==========================================
# FLASK APP CORE SYSTEM INITIALIZATION
# ==========================================

# Instantiates the Flask container application.
app = Flask(__name__)
# 'current_path' captures the current script working folder route.
current_path = os.getcwd()
# Cryptographic validation key securing browser session state transactions.
app.secret_key = os.getenv("SECRET_KEY", "fallback-if-env-fails")
# 'message_cache' maps in-memory context records to active chat sessions for performance caching.
message_cache = dict()

# ==========================================
# CORS SECURITY PROTOCOLS CONFIGURATION
# ==========================================

# 'ALLOWED_ORIGINS' controls which client IP addresses and domain hosts are cleared to query this backend.
ALLOWED_ORIGINS = [
    "http://localhost:3000",
    "http://localhost:3001",
    "https://dev.raims.chat",
]

# Binds allowed CORS request origins, enabling credentials and customizing authorized headers.
CORS(
    app,
    resources={r"/*": {
        "origins": ALLOWED_ORIGINS,
        "allow_headers": ["Authorization", "Content-Type"]
    }},
    supports_credentials=True
)

# Global placeholder containing the initialized FAISS vectorstore reference.
vectorstore = None

# Registers structural application log file targets.
file_handler = logging.FileHandler('app.log')
file_handler.setLevel(logging.DEBUG)

# Formatting logs to include standard temporal and severity level tags.
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

# Minimizes logging overhead coming from internal Flask routing calls.
logging.getLogger('werkzeug').setLevel(logging.WARNING)

# Global trace variable keeping record of previous system statements.
previous_response = None

# ==========================================
# HELPER & AUTHENTICATION UTILITY FUNCTIONS
# ==========================================

# Function: create_jwt_token
# Args: user_uid (string)
# Returns: JWT String containing encoded session properties.
def create_jwt_token(user_uid):
    """Create a simple JWT token for the user"""
    payload = {
        'uid': user_uid,
        'exp': int(time.time()) + (24 * 60 * 60),  # Token expires 24 hours from issuance.
        'iat': int(time.time())
    }
    secret_key = app.secret_key
    return jwt.encode(payload, secret_key, algorithm='HS256')


# Function: verify_jwt_token
# Args: token (string)
# Returns: Decrypted payload dictionary if valid, otherwise None.
def verify_jwt_token(token):
    """Verify our custom JWT token"""
    try:
        secret_key = app.secret_key
        payload = jwt.decode(token, secret_key, algorithms=['HS256'])
        return payload
    except jwt.ExpiredSignatureError:
        return None
    except jwt.InvalidTokenError:
        return None


# Function: verify_user_token
# Args: token (string)
# Returns: User structural document containing user IDs or None.
def verify_user_token(token):
    """Verify our custom JWT token and return user info"""
    try:
        payload = verify_jwt_token(token)
        if payload:
            return {'uid': payload['uid']}
        return None
    except Exception as e:
        logger.error(f"Error verifying token: {e}")
        return None


# Function: get_user_from_request
# Args: None
# Returns: Authorized user context dictionary or None.
def get_user_from_request():
    """Extract and verify user from request headers"""
    auth_header = request.headers.get('Authorization')
    if not auth_header or not auth_header.startswith('Bearer '):
        return None

    id_token = auth_header.split('Bearer ')[1]
    return verify_user_token(id_token)


# Function: verify_google_token
# Args: token (string)
# Returns: Verified user profile document from Google verification services or None.
def verify_google_token(token):
    """Verify Google OAuth token and return user info"""
    try:
        print(f"=== GOOGLE TOKEN VERIFICATION ===")
        print(f"Token: {token[:50]}..." if token else "NO TOKEN")
        print(f

In [2]:
# @title AI prompt cell

import ipywidgets as widgets
from IPython.display import display, HTML, Markdown,clear_output
from google.colab import ai

dropdown = widgets.Dropdown(
    options=[],
    layout={'width': 'auto'}
)

def update_model_list(new_options):
    dropdown.options = new_options
update_model_list(ai.list_models())

text_input = widgets.Textarea(
    placeholder='Ask me anything....',
    layout={'width': 'auto', 'height': '100px'},
)

button = widgets.Button(
    description='Submit Text',
    disabled=False,
    tooltip='Click to submit the text',
    icon='check'
)

output_area = widgets.Output(
     layout={'width': 'auto', 'max_height': '300px','overflow_y': 'scroll'}
)

def on_button_clicked(b):
    with output_area:
        output_area.clear_output(wait=False)
        accumulated_content = ""
        for new_chunk in ai.generate_text(prompt=text_input.value, model_name=dropdown.value, stream=True):
            if new_chunk is None:
                continue
            accumulated_content += new_chunk
            clear_output(wait=True)
            display(Markdown(accumulated_content))

button.on_click(on_button_clicked)
vbox = widgets.GridBox([dropdown, text_input, button, output_area])

display(HTML("""
<style>
.widget-dropdown select {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
.widget-textarea textarea {
    font-size: 18px;
    font-family: "Arial", sans-serif;
}
</style>
"""))
display(vbox)


GridBox(children=(Dropdown(layout=Layout(width='auto'), options=('google/gemini-3.5-flash-lite', 'google/gemin…